# Week 2 Day 2: what does the memory add, and how should it be built?

*Feyisa Diba's version of the Week 2 Day 2 assignment. My results, what I concluded and how they compare with other students' runs are in "What I found" at the end.*

The assignment asks three things of the lab's memory agent:

1. **Challenge:** replace Qdrant with a simple word lookup over `data/knowledge.json`. How do the scores compare with searching by meaning?
2. **A memory about each customer:** a Markdown file of notes for each one, with a tool to read them and a tool to add to them, and a few notes written by hand to start with. With `N_TIMES` above 1, what should happen to the notes between runs?
3. **Stretch:** move the retrieval into a research sub-agent that has `qdrant-find` and the notes tools, and let the UnderCover agent call it as a tool.

Other students have answered all three, and their answers disagree. The word lookup beat Qdrant in one notebook, tied in another and lost in a third, and the sub-agent helped in some and hurt in others. Most ran each experiment once or twice, and one run of 25 claims is noisy. As on Day 1, this notebook tries to separate the differences from the noise:

- **Three runs of every experiment** on all 25 claims, with GPT-6 Luna at `low` as in the lab, and a **paired bootstrap** for each comparison.
- **Every claim counts.** Langfuse leaves a claim out of a run if its task raises an error, so the task never raises: it tries again, and a claim that still fails counts as a referral.
- **Within the rate limit.** OpenRouter allows new accounts 20 requests a minute for GPT-6 Luna, and the memory agents make more calls a claim than Day 1's. Every model call, the sub-agent's too, goes through LangChain's rate limiter at 17 a minute. The time a claim takes leaves out the time spent waiting for it.
- **A retrieval test with no model calls.** Every query the agents wrote is sent to both searches again, which shows whether the word lookup would have found what Qdrant found, and the reverse.
- **Starting notes that test something:** seven notes of three kinds, including three misleading ones that the policy wording says should change nothing.
- **The question about `N_TIMES`, answered with data:** two more runs that keep the notes from the runs before instead of resetting them.

The agents, tools and runs are in [week2day2_memory.py](week2day2_memory.py), so the experiments can also run from the command line. Each finished run is saved to `week2day2_memory_results.csv` and loaded from there the next time, so once the runs are done the analysis can be run again for free.

In [ ]:
import inspect
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import display
from langchain_mcp_adapters.tools import load_mcp_tools

from week2day2_memory import (
    DESCRIPTIONS,
    MEMORY,
    MEMORY_LINE,
    NOTES_LINE,
    RESEARCH_INSTRUCTIONS,
    RESEARCH_LINE,
    SEED_NOTES,
    add_customer_note,
    build_agents,
    claims,
    dataset_items,
    is_right,
    knowledge,
    labels,
    langfuse,
    mistake_cost,
    qdrant_client,
    read_customer_notes,
    research,
    run_once,
    run_scores,
    share,
    word_lookup,
    word_search,
)

pd.set_option("display.max_colwidth", None)


def show(*things):
    """Print the code of functions and tools from week2day2_memory.py."""
    for thing in things:
        print(inspect.getsource(getattr(thing, "coroutine", None) or getattr(thing, "func", None) or thing))

## Connect to OpenRouter and Langfuse

`week2day2_memory.py` loads the keys from the `.env` file in the `undercover` folder, as the lab does.

In [ ]:
print("OpenRouter API key found" if os.getenv("OPENROUTER_API_KEY") else "No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")
print("Connected to Langfuse" if langfuse.auth_check() else "No Langfuse keys found. Add them to the .env file")

## The claims and the facts they need

The lab's 25 even harder claims. Each of 19 turns on one fact from the knowledge base: when a storm was named or a strike announced, what a device is, how deep a dive site goes, what grade a river's rapids are, or what an activity operator does. The other 6 need none.

In [ ]:
needs = pd.DataFrame([
    {
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "handler": labels[claim["claim_id"]]["decision"],
        "payout": labels[claim["claim_id"]]["payout"],
        "needs": ", ".join(labels[claim["claim_id"]]["needs"]) or "-",
    }
    for claim in claims
])
print(f"{(needs['needs'] != '-').sum()} of the {len(claims)} claims need one of the {len(knowledge)} facts")
needs

## The memory

As in the lab, Qdrant runs as an MCP server with `uvx`, and each `qdrant-find` starts the server, searches and stops it. One thing is different. The lab has an assistant store the facts. Here they go in directly, one `qdrant-store` call for each fact, word for word, as the lab's free version does. The word lookup and Qdrant then search exactly the same 33 texts, so the only difference between them is how they search.

`build_agents` stores the facts once, then makes the agent for each experiment.

In [ ]:
agents = await build_agents()
list(agents)

## Challenge 1: a word lookup

The word lookup splits the query into words, in lower case. It leaves out single letters and the commonest words, but keeps numbers, so that "Lumora Drift 3" isn't "Lumora Drift 2". It ranks the facts by how many of the query's words each contains, and returns up to 10.

It answers in the same format as `qdrant-find`, so the agent sees the same kind of result. Its agent has the same instructions as the Qdrant agent, including the lab's line about searching its memory. The tool is called `word_lookup`, and `decide` picks out the results of both tools by their names.

In [ ]:
show(word_search, word_lookup)

The two searches side by side, on a few queries. `qdrant_search` keeps one session of the server open, so that many queries are quick. The lab's example, a gadget for watching films on a wall, shares no words with the Halden Arc's fact.

In [ ]:
def entries(text):
    """The facts in a qdrant-find result, nearest first."""
    return [re.search(r"<content>(.*)</content>", line, re.S).group(1) for line in json.loads(text)[1:]]


async def qdrant_search(queries):
    """Qdrant's results for each query, nearest first, from one session of the server."""
    async with qdrant_client(MEMORY).session("memory") as session:
        find = [memory_tool for memory_tool in await load_mcp_tools(session) if memory_tool.name == "qdrant-find"][0]
        return {query: entries((await find.ainvoke({"query": query}))[0]["text"]) for query in queries}


def top(facts, count=3):
    """The start of each of the first few facts."""
    return " | ".join(fact[:40] for fact in facts[:count]) or "nothing"


EXAMPLES = ["Halden Arc", "a gadget for watching films on a wall", "Lumora Drift 3", "Hurricane Marisol Bahamas", "how deep is the dive site"]
examples = await qdrant_search(EXAMPLES)
pd.DataFrame([{"query": query, "by meaning (Qdrant)": top(examples[query]), "by words": top(word_search(query))} for query in EXAMPLES])

## Challenge 2: a memory about each customer

Each customer gets a Markdown file, `CUS-3301.md` and so on, named by the claim's `customer_id`. `read_customer_notes` returns the file, and `add_customer_note` adds a dated line to it. The agent once filed a note under a claim id in VerBodh's run, so `add_customer_note` turns down an id that isn't a customer's.

The agent has the Qdrant agent's tools and instructions, with one more line:

In [ ]:
print(NOTES_LINE)
show(read_customer_notes, add_customer_note)

### The starting notes

Each customer has only one claim in this set, so a note can't carry from one of their claims to the next within a run. Starting notes that only repeat the claim would test nothing, so the seven notes in `SEED_NOTES`, in `week2day2_memory.py`, are of three kinds, written as the handlers' notes from phone calls:

- **Neutral:** a contact preference, and two calls that match the claim. They should change nothing.
- **Helpful:** a handler told the customer what the wording needs. It points the same way as the wording.
- **Misleading:** pressure that the wording says should change nothing. A loyal, upset customer on a claim the wording excludes; a rude customer on a claim it covers; and a note that the police report "has arrived" when it isn't attached to the claim. The handler referred that claim because the report is missing, and a phone call isn't a document.

None of them says what the handler decided.

In [ ]:
NOTE_KINDS = {
    "CUS-3305": ("misleading", "a loyal, upset customer, on a claim the wording excludes"),
    "CUS-3307": ("neutral", "a call that matches the claim"),
    "CUS-3319": ("misleading", "a rude customer, on a claim the wording covers"),
    "CUS-3320": ("neutral", "a contact preference"),
    "CUS-3321": ("neutral", "a call that matches the claim"),
    "CUS-3324": ("misleading", "a police report that 'has arrived' but isn't attached"),
    "CUS-3325": ("helpful", "the handler said the itemized bill is needed"),
}
claim_of = {claim["customer_id"]: claim["claim_id"] for claim in claims}

seeds = pd.DataFrame([
    {
        "customer": customer_id,
        "claim_id": claim_of[customer_id],
        "handler": labels[claim_of[customer_id]]["decision"],
        "kind": NOTE_KINDS[customer_id][0],
        "what it tests": NOTE_KINDS[customer_id][1],
        "note": text.split("\n", 2)[2].strip(),
    }
    for customer_id, text in SEED_NOTES.items()
])
seeds

### What should happen to the notes between runs?

Every classmate who answered says to reset them, and I agreed before the runs. In Andras Nemes's words: **reset them to the starting notes before every run**, as the lab clears `recorded`. Otherwise run 2 reads what run 1 wrote about the same claim, such as "approved $562.95". Each run of an experiment should be a repeat of the same test, so that the runs show how much the agent varies.

In production the notes should carry on, because that's the point of a memory. So the experiment **Qdrant + customer notes** resets them before every run, and **Notes kept** shows what happens if they aren't reset. Andras and Darko Martinovic each ran one run like that. Here, run 2 of Notes kept starts from the notes that run 1 of Qdrant + customer notes left, and its run 3 from the notes that its run 2 left. That's a chain of three runs with no reset, compared with runs 2 and 3 of the experiment that resets, to see whether the effect grows.

The working copy of the notes for each run is a fresh folder. The notes each claim added are saved with the run, so a later run of Notes kept can rebuild the notes it starts from.

## Stretch: a research sub-agent

The research assistant has `qdrant-find` and both notes tools. The UnderCover agent keeps `record_decision` and `calculate_payout`, loses the memory and notes tools, and gets `research`, which asks the assistant and returns its reply.

Three things need care, and Andras Nemes found all three:

- **The scores would miss the sub-agent's work.** Its searches and model calls happen in its own conversation, so `decide` wouldn't see them. `research` logs them, so `found_fact` counts its searches and the cost includes its calls.
- **The sub-agent sees only the request.** It never sees the claim or the policy, and the storm and strike claims turn on dates. So the request asks for "any dates that matter", the assistant quotes what it finds word for word, and `relayed_fact` checks that the needed fact reached the UnderCover agent, not only that a search found it.
- **The sub-agent's calls nest inside the UnderCover agent's trace in Langfuse** with no extra code, because LangChain passes the callbacks down.

In [ ]:
print(RESEARCH_LINE)
print(RESEARCH_INSTRUCTIONS)
show(research)

## The experiments

| Experiment | The UnderCover agent's tools | Its extra instructions |
|---|---|---|
| Payout tool | `record_decision`, `calculate_payout` | none (the lab's Experiment 1) |
| Qdrant | and `qdrant-find` | search the memory (the lab's Experiment 2) |
| Word lookup | and `word_lookup` | search the memory |
| Qdrant + customer notes | and `qdrant-find`, `read_customer_notes`, `add_customer_note` | search the memory, read and add to the notes |
| Notes kept | the same, without resetting the notes | the same |
| Research sub-agent | and `research` | ask the research assistant, and have it add a note |

`N_TIMES` runs of each, except Notes kept, whose run 1 is run 1 of Qdrant + customer notes. `TINY = True` runs the 5 tiny claims for a quick check, isn't saved, and leaves out Notes kept.

The runs take turns, as on Day 1: run 1 of each experiment, then run 2 of each, then run 3. Then a change in the provider's speed during the session can't look like a difference between experiments.

The 17 runs make about 2,000 model calls, so at 17 a minute they take about two hours. I ran them from the command line in this same order, with `uv run python week2day2_memory.py "Payout tool:1" "Qdrant:1" ...`, and this cell loads them from `week2day2_memory_results.csv`.

My first attempt ran the six experiments as five processes side by side, to finish sooner. Together they went far over the limit, and claims failed with 429 errors, so I stopped them and threw those runs away. The rate limit is for the account, not for each process.

In [ ]:
TINY = False
N_TIMES = 3

items = dataset_items()
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

PLAN = [(name, run) for run in range(1, N_TIMES + 1) for name in DESCRIPTIONS if not (name == "Notes kept" and (run == 1 or TINY))]
results = pd.concat([run_once(name, run, agents, items) for name, run in PLAN], ignore_index=True)
EXPERIMENTS = {name: results[results["experiment"] == name] for name in DESCRIPTIONS}
print(f"{len(results)} answers from {results.groupby(['experiment', 'run']).ngroups} runs, costing ${results['llm_cost'].sum():.3f} in model calls")

## The results

Each figure is the average of the runs. `found_fact` is the share of the 19 claims that need a fact where the searches brought it back. `relayed_fact` is the share where the research assistant passed it on. `refused_changes` counts the claims where the agent tried to change a decision it had already recorded, which `record_decision` refuses because decisions are final. Then the claims each run got right, out of 25.

In [ ]:
def runs_of(table):
    """The scores of each run of an experiment, one row a run."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")])


summary = pd.DataFrame({name: runs_of(table).mean() for name, table in EXPERIMENTS.items()}).T
display(summary.round(3))
results.groupby(["experiment", "run"])["right"].sum().unstack().reindex(DESCRIPTIONS)

The kinds of mistake, added up over the runs. Notes kept has two runs, and the others have three.

In [ ]:
def mistake_kind(row):
    """What kind of mistake an answer is, against the handler's."""
    if row.right:
        return "right"
    if row.agent == row.handler:
        return "wrong amount"
    if row.agent == "decline":
        return "wrongful decline"
    if row.agent == "approve":
        return "wrongful approval"
    return "needless referral"


results["kind"] = [mistake_kind(row) for row in results.itertuples()]
EXPERIMENTS = {name: results[results["experiment"] == name] for name in DESCRIPTIONS}
results.groupby("experiment")["kind"].value_counts().unstack(fill_value=0).reindex(DESCRIPTIONS)

## Is the difference real?

As on Day 1: every experiment decided the same 25 claims, so each comparison is made claim by claim. Each claim's average over its runs in one experiment is taken from its average in the other. A bootstrap resamples the 25 claims 10,000 times to give a 95% interval for each difference. When the interval includes zero, these runs can't tell the two apart.

The comparisons:

- **Payout tool to Qdrant:** what the memory adds, the lab's question.
- **Qdrant to Word lookup:** Challenge 1.
- **Qdrant to Qdrant + customer notes:** Challenge 2, adding the notes.
- **Qdrant + customer notes to Research sub-agent:** the stretch. The sub-agent has the same tools, moved into a second agent.
- **Qdrant + customer notes to Notes kept, runs 2 and 3:** whether to reset the notes.

In [ ]:
MEASURES = {  # what we report: (the column, and the scale from a claim's average to the figure, where None means the number of claims)
    "accuracy (%)": ("right", 100),
    "cost of mistakes a run ($)": ("mistake_cost", None),
    "model cost a run ($)": ("llm_cost", None),
    "seconds a claim": ("seconds", 1),
    "model calls a claim": ("model_calls", 1),
}
def resamples(count):
    """10,000 bootstrap resamples of the claims' positions, the same every time."""
    return np.random.default_rng(2026).integers(0, count, size=(10_000, count))


def per_claim(table):
    """Each claim's average over the runs."""
    return table.groupby("claim_id")[[column for column, _ in MEASURES.values()]].mean()


def compare(first, second, runs=None):
    """second minus first on each measure, with a paired bootstrap 95% interval. runs keeps only those runs of both."""
    tables = [EXPERIMENTS[first], EXPERIMENTS[second]]
    if runs:
        tables = [table[table["run"].isin(runs)] for table in tables]
    a = per_claim(tables[0])
    b = per_claim(tables[1]).loc[a.index]
    rows = {}
    for measure, (column, scale) in MEASURES.items():
        scale = scale or len(a)
        differences = (b[column] - a[column]).to_numpy(dtype=float)
        resampled = np.nanmean(differences[resamples(len(a))], axis=1) * scale
        low, high = np.percentile(resampled, [2.5, 97.5])
        rows[measure] = {
            "first": np.nanmean(a[column]) * scale,
            "second": np.nanmean(b[column]) * scale,
            "difference": np.nanmean(differences) * scale,
            "95% from": low,
            "95% to": high,
            "clear?": "yes" if low > 0 or high < 0 else "no",
        }
    return pd.DataFrame.from_dict(rows, orient="index")


PAIRS = [
    ("Payout tool", "Qdrant", None),
    ("Qdrant", "Word lookup", None),
    ("Qdrant", "Qdrant + customer notes", None),
    ("Qdrant + customer notes", "Research sub-agent", None),
    ("Qdrant + customer notes", "Notes kept", [2, 3]),
]
pd.concat({f"{first} -> {second}" + (" (runs 2 and 3)" if runs else ""): compare(first, second, runs) for first, second, runs in PAIRS}).round(3)

## Claim by claim

The claims that any experiment got wrong in at least one run, with the runs each got right.

In [ ]:
right_of_runs = results.groupby(["claim_id", "experiment"])["right"].agg(lambda right: f"{right.sum()}/{len(right)}").unstack().reindex(columns=list(DESCRIPTIONS))
always_right = results.groupby("claim_id")["right"].all()
by_claim = needs.set_index("claim_id")[["handler", "payout", "needs"]].join(right_of_runs)
by_claim[~always_right]

Every wrong answer from the experiments with a memory, with the agent's rationale. The payout tool's mistakes without a memory are the lab's, and are left out.

In [ ]:
COLUMNS = ["claim_id", "experiment", "run", "handler", "handler_payout", "agent", "agent_payout", "found_fact", "refused_changes", "rationale"]
results[~results["right"] & (results["experiment"] != "Payout tool")][COLUMNS].sort_values(["claim_id", "experiment", "run"])

## Decisions are final: the second thought

Many wrong declines above share a pattern, which Oleg Mosiichuk found in his run. The agent writes its reasoning into the rationale as it records the decision. It starts from the exclusion for events known before the policy was bought, then catches its own mistake with the dates halfway through ("before the policy was purchased on July 28? Actually, August 5 is after purchase"), but the decision it records is still decline. On its next turn it tries to record the right answer, and `record_decision` refuses, because decisions are final.

The fact was found every time. The mistake is comparing two dates, and then a rule that keeps the first answer.

This scores the same answers again, with no model calls, as if the agent's last attempt to record a decision had been accepted, as in Oleg's version, where a decision is a draft until the agent finishes. It's an estimate: an agent that knows it can revise might also behave differently.

In [ ]:
def last_attempt(row):
    """The agent's decision and payout if its last attempt to record one had been accepted."""
    changes = json.loads(row.refused_changes)
    if not changes:
        return {"decision": row.agent, "payout": row.agent_payout}
    decision, payout = changes[-1].split()
    return {"decision": decision, "payout": float(payout)}


second = []
for row in results.itertuples():
    answer, expected = last_attempt(row), {"decision": row.handler, "payout": row.handler_payout}
    second.append({"right_if_revised": is_right(answer, expected), "mistake_cost_if_revised": mistake_cost(answer, expected)})
results = results.join(pd.DataFrame(second, index=results.index))
EXPERIMENTS = {name: results[results["experiment"] == name] for name in DESCRIPTIONS}

runs = results.groupby("experiment")["run"].nunique()
pd.DataFrame({
    "tried to change, a run": results.groupby("experiment")["refused_changes"].apply(lambda changes: (changes != "[]").sum()) / runs,
    "accuracy": results.groupby("experiment")["right"].mean(),
    "accuracy if revised": results.groupby("experiment")["right_if_revised"].mean(),
    "cost of mistakes a run": results.groupby("experiment")["mistake_cost"].sum() / runs,
    "cost of mistakes a run if revised": results.groupby("experiment")["mistake_cost_if_revised"].sum() / runs,
}).reindex(DESCRIPTIONS).round(3)

How often a rationale corrects itself partway through, with "Actually", "Wait" or "Correction", and how many of those answers are wrong.

In [ ]:
corrects_itself = results["rationale"].str.contains(r"\bActually\b|\bWait\b|Correction", case=False)
results[corrects_itself].groupby("experiment")["right"].agg(rationales="size", wrong=lambda right: (~right).sum()).reindex(DESCRIPTIONS).fillna(0).astype(int)

## Challenge 1: searching by words against searching by meaning

`found_fact` above shows whether each experiment's searches found the fact. To see whether the other search would have found it with the same queries, every query any agent wrote is sent to both searches again, with no model calls. For each claim in each run, the best rank of the needed fact over that run's queries gives three retrieval scores, as on the slides: whether it was found in the top 10, whether it was ranked first, and the mean reciprocal rank (MRR), where first scores 1, second 0.5, and a miss 0.

In [ ]:
searched = results[results["experiment"] != "Payout tool"]
all_queries = sorted({query for queries in searched["queries"] for query in json.loads(queries)} | {claim["summary"] for claim in claims})
SEARCHES = {"by meaning": await qdrant_search(all_queries), "by words": {query: word_search(query) for query in all_queries}}
print(f"{len(all_queries)} distinct queries")


def best_rank(queries, search, need):
    """The best position of the needed fact in the results of a set of queries, from 1, or NaN if none found it."""
    return min((position for query in queries for position, fact in enumerate(search[query], 1) if need in fact), default=np.nan)


def retrieval_scores(ranks):
    """Found in the top 10, ranked first, and the mean reciprocal rank."""
    return pd.Series({"found in top 10": ranks.notna().mean(), "ranked first": (ranks == 1).mean(), "MRR": (1 / ranks).fillna(0).mean()})


replay = pd.DataFrame([
    {"experiment": row.experiment, "run": row.run, "claim_id": row.claim_id, "search": name, "rank": best_rank(json.loads(row.queries), search, labels[row.claim_id]["needs"][0])}
    for row in searched.itertuples()
    if labels[row.claim_id]["needs"]
    for name, search in SEARCHES.items()
])
replay.groupby(["experiment", "search"])["rank"].apply(retrieval_scores).unstack().round(3)

**One query that isn't the agent's.** A plain RAG pipeline, with no agent, might search with the claim's summary. Arthur JZL tried this, and found the fact in the top 10 for 74% of claims by words and 89% by meaning. With the summary as the query, the summary names the thing for some claims and only describes it for others.

In [ ]:
summary_ranks = pd.DataFrame([
    {"claim_id": claim["claim_id"], "summary": claim["summary"], "search": name, "rank": best_rank([claim["summary"]], search, labels[claim["claim_id"]]["needs"][0])}
    for claim in claims
    if labels[claim["claim_id"]]["needs"]
    for name, search in SEARCHES.items()
])
display(summary_ranks.groupby("search")["rank"].apply(retrieval_scores).unstack().round(3))
summary_ranks.pivot(index=["claim_id", "summary"], columns="search", values="rank")

## Challenge 2: the customer notes

Did the agent use the notes? The share of claims where it read them, and where it added one.

In [ ]:
NOTED = ["Qdrant + customer notes", "Notes kept", "Research sub-agent"]
noted = results[results["experiment"].isin(NOTED)]
noted.assign(read=noted["notes_read"] > 0, added=noted["notes_added"] != "[]").groupby("experiment")[["read", "added"]].mean().round(3)

### The claims with starting notes

The runs each experiment got right on the seven claims with a starting note. Qdrant has no notes, so it shows how the same agent does on these claims without them.

In [ ]:
seed_right = right_of_runs.loc[seeds["claim_id"], ["Qdrant", "Qdrant + customer notes", "Research sub-agent"]]
seeds.set_index("claim_id")[["customer", "handler", "kind", "what it tests"]].join(seed_right)

The answers on the three claims with a misleading note, in the experiments that read the notes, with the rationale. Does the rationale mention the note?

In [ ]:
MISLEADING = seeds.loc[seeds["kind"] == "misleading", "claim_id"]
misled = results[results["claim_id"].isin(MISLEADING) & results["experiment"].isin(["Qdrant + customer notes", "Research sub-agent"])]
misled[["claim_id", "experiment", "run", "handler", "agent", "agent_payout", "right", "rationale"]].sort_values(["claim_id", "experiment", "run"])

### The notes the agent wrote

The notes from run 1 of Qdrant + customer notes, one for each claim. A quick check: does the note mention the decision the agent recorded? It looks for "approv", "declin" or "refer" in the note. Then the notes in any run that don't.

In [ ]:
STEMS = {"approve": "approv", "decline": "declin", "refer": "refer"}


def notes_text(notes):
    """The notes added on one claim, as one piece of text."""
    return " ".join(line for _, line in json.loads(notes))


written = noted.assign(note=noted["notes_added"].map(notes_text))
written = written[written["note"] != ""]
written = written.assign(mentions_decision=[STEMS[row.agent] in row.note.lower() for row in written.itertuples()])
print(written.groupby("experiment")["mentions_decision"].mean().round(3))
first_run = written[(written["experiment"] == "Qdrant + customer notes") & (written["run"] == 1)]
display(first_run[["claim_id", "agent", "agent_payout", "mentions_decision", "note"]])
written[~written["mentions_decision"]][["claim_id", "experiment", "run", "agent", "agent_payout", "note"]]

### Resetting the notes, or keeping them

Runs 2 and 3 of the two experiments, compared above for accuracy and cost. Two more checks:

- **How often a run repeats the answer of the run before it in its chain**, as the same decision and payout. With a reset, two runs agree only as often as the agent is consistent. If keeping the notes makes them agree more often, the agent is copying its earlier answer.
- **Rationales that point to an earlier decision**, such as "already", "previous" or "duplicate".

In [ ]:
reset, kept = EXPERIMENTS["Qdrant + customer notes"], EXPERIMENTS["Notes kept"]


def answers_of(table, run):
    """Each claim's decision and payout in one run."""
    one = table[table["run"] == run].set_index("claim_id").sort_index()
    return one["agent"] + " " + one["agent_payout"].round(2).astype(str)


def repeated(later, earlier):
    """The share of claims where a run gave the same answer as an earlier one."""
    return (later == earlier.loc[later.index]).mean()


pd.DataFrame({
    "reset": [repeated(answers_of(reset, 2), answers_of(reset, 1)), repeated(answers_of(reset, 3), answers_of(reset, 2))],
    "kept": [repeated(answers_of(kept, 2), answers_of(reset, 1)), repeated(answers_of(kept, 3), answers_of(kept, 2))],
}, index=["run 2, against run 1", "run 3, against run 2"]).round(3)

In [ ]:
EARLIER = r"earlier|previous|prior|already|duplicate|existing|resubmi"
pointing_back = pd.concat([reset[reset["run"] > 1], kept])
pointing_back = pointing_back[pointing_back["rationale"].str.contains(EARLIER, case=False)]
pointing_back[["claim_id", "experiment", "run", "handler", "agent", "agent_payout", "right", "rationale"]].sort_values(["experiment", "claim_id", "run"])

Keeping the notes also carried something useful, on CLM-1304. Qdrant + customer notes declined it in all three runs, with the slip over the dates. In its run 1 the agent's correction was refused, so it wrote the correction into the customer's note, as Hugo Volckaert's agent did on CUS-3304. Notes kept read that note in runs 2 and 3, and approved. Here are the note each run added and the answer each run gave.

In [ ]:
chain = pd.concat([reset[reset["run"] == 1], kept])
chain = chain[chain["claim_id"] == "CLM-1304"]
chain.assign(note=chain["notes_added"].map(notes_text))[["experiment", "run", "agent", "agent_payout", "right", "note"]]

## Stretch: the research sub-agent

How much of the work moved into the sub-agent, and whether the facts got through.

In [ ]:
sub = EXPERIMENTS["Research sub-agent"]
pd.Series({
    "found_fact": share(sub["found_fact"]),
    "relayed_fact": share(sub["relayed_fact"]),
    "research calls a claim": sub["research_calls"].mean(),
    "share of model calls in the sub-agent": sub["sub_model_calls"].sum() / sub["model_calls"].sum(),
    "model cost a run": sub["llm_cost"].sum() / sub["run"].nunique(),
    "seconds a claim": sub["seconds"].mean(),
}).round(3)

The storm and strike claims turn on whether a fact came before the policy was bought. Hugo Volckaert's sub-agent fixed his wrongful declines on them, and he suggested that the research makes comparing the dates a step of its own. Andras Nemes saw his agent learn to pass on the policy date. The sub-agent sees only the request, so it can answer that only when the request gives the date the policy was bought. Over the three runs: did the UnderCover agent's request give that date, did the reply say "before" or "after" the purchase, and was the claim decided right?

In [ ]:
DATED = ["CLM-1301", "CLM-1302", "CLM-1303", "CLM-1304", "CLM-1305", "CLM-1306"]
dated = sub[sub["claim_id"].isin(DATED)]


def lookups(row):
    """The requests to the research assistant on one claim, and its replies, leaving out the notes it was asked to add."""
    return [exchange for exchange in json.loads(row.research) if not exchange["request"].lower().startswith("add")]


dated = dated.assign(
    gave_purchase_date=[any(re.search(r"purchas|bought", exchange["request"], re.I) for exchange in lookups(row)) for row in dated.itertuples()],
    reply_says_before_or_after=[
        any(re.search(r"\b(before|after|predat|postdat)\w*\b.{0,40}(purchas|bought|policy)", exchange["reply"], re.I) for exchange in lookups(row))
        for row in dated.itertuples()
    ],
)
dated.groupby(["gave_purchase_date", "reply_says_before_or_after"])["right"].agg(claims="size", right="sum")

## What I found

Three runs of each experiment on the 25 even harder claims, and two of Notes kept, with GPT-6 Luna at `low`, on 21 November 2026. The 17 runs made 2,059 model calls and cost $0.264. The outputs above are cleared, so the tables here hold the results.

### The answers

1. **Searching by words or by meaning? Here the word lookup searches as well as Qdrant, and it's faster.**
   - With the agents' own queries, both found the needed fact in the top 10 on every claim in every run.
   - The agent with the word lookup scored 82.7% against Qdrant's 86.7%. The interval, -15 to +5 points, includes zero.
   - It's 3.3 seconds a claim faster, because no server has to start.
   - Searching by meaning matters when the query describes the thing instead of naming it. With the claim's summary as the query, the word lookup found the fact for only 73.7% of claims, against 89.5%.
2. **Customer notes did no harm and no measurable good. Reset them before every run.**
   - The agent read and added to the notes on every claim.
   - None of the three misleading notes changed a decision.
   - The notes added 5.3 points of accuracy, which isn't clear, for 21% more model cost.
   - With the notes kept between runs, each run read the last one's answer to the same claim. Then it referred claims it had approved before, to avoid paying twice.
3. **The research sub-agent works, at a clear cost, for no clear gain.**
   - It found the fact and passed it on every time.
   - Accuracy was 93.3% against 92.0%, which isn't clear.
   - It cost 67% more in model calls and took 5.7 seconds more a claim.

The lab's own question: **the memory raises accuracy from 54.7% to 86.7%, a clear gain of 32 points, but it doesn't lower the cost of mistakes.** Oleg Mosiichuk saw this first, in his run of the lab. Most of what's left is a slip over dates, which the agent then tries to correct. `record_decision` refuses the correction, because decisions are final. Accepting the agent's last attempt would cut Qdrant's cost of mistakes from $2,614 a run to $614. That's a bigger gain than any of the three changes the assignment asks for.

### The averages of three runs

Notes kept has two runs. The figures in brackets are the lowest and highest run.

| | Payout tool | Qdrant | Word lookup | Qdrant + notes | Notes kept | Sub-agent |
|---|---|---|---|---|---|---|
| Accuracy | 54.7% (48 to 60) | 86.7% (80 to 92) | 82.7% (80 to 84) | 92.0% (88 to 96) | 78.0% (76 to 80) | **93.3%** (88 to 96) |
| Decision accuracy | 58.7% | 88.0% | 84.0% | 92.0% | 78.0% | **94.7%** |
| Kappa | 0.274 | 0.781 | 0.709 | 0.846 | 0.635 | **0.895** |
| Cost of mistakes a run | $2,081 | $2,614 | $4,578 | $2,033 | $1,000 | **$578** |
| Wrongful declines a run | 0.33 | 1.67 | 3.00 | 1.33 | 0.50 | 0.33 |
| Straight-through rate | 61% | 87% | 88% | 89% | 72% | 88% |
| Searches found the fact | - | 100% | 100% | 100% | 100% | 100% |
| Model cost a run | $0.0094 | $0.0134 | $0.0128 | $0.0162 | $0.0138 | $0.0271 |
| Seconds a claim (median) | 5.4 (4.8) | 10.7 (9.8) | 7.3 (6.5) | 11.8 (11.4) | 12.2 (11.5) | 17.4 (17.3) |
| Model calls a claim | 2.6 | 3.7 | 3.7 | 4.5 | 4.4 | 10.0 |
| Input tokens a claim | 8,710 | 14,005 | 13,429 | 18,009 | 17,908 | 20,197 |
| Failed claims | 0 | 0 | 0 | 0 | 0 | 0 |

The handlers refer 2 of the 25 claims, so their straight-through rate is 92%.

### The comparisons, claim by claim

| Comparison | Measure | Difference | 95% interval | Clear? |
|---|---|---|---|---|
| Payout tool to Qdrant | accuracy | +32.0 points | +10.7 to +52.0 | yes |
| | cost of mistakes a run | +$533 | -$2,537 to +$4,201 | no |
| | model cost a run | +$0.004 | +$0.003 to +$0.005 | yes |
| | seconds a claim | +5.2 | +4.6 to +5.9 | yes |
| Qdrant to Word lookup | accuracy | -4.0 points | -14.7 to +5.3 | no |
| | cost of mistakes a run | +$1,964 | -$7 to +$4,495 | no, just |
| | seconds a claim | -3.3 | -4.1 to -2.6 | yes |
| Qdrant to Qdrant + notes | accuracy | +5.3 points | -2.7 to +13.3 | no |
| | cost of mistakes a run | -$581 | -$2,517 to +$969 | no |
| | model cost a run | +$0.003 | +$0.002 to +$0.003 | yes |
| | seconds a claim | +1.1 | +0.3 to +1.8 | yes |
| Qdrant + notes to Sub-agent | accuracy | +1.3 points | -8.0 to +12.0 | no |
| | cost of mistakes a run | -$1,455 | -$5,033 to +$1,067 | no |
| | model cost a run | +$0.011 | +$0.010 to +$0.012 | yes |
| | seconds a claim | +5.7 | +4.0 to +7.4 | yes |
| Qdrant + notes to Notes kept, runs 2 and 3 | accuracy | -12.0 points | -30.0 to +6.0 | no |
| | cost of mistakes a run | -$1,300 | -$5,175 to +$1,775 | no |

### Where the cost of mistakes comes from

The cost of mistakes a run, by kind of mistake:

| | Needless referrals | Wrong amounts | Wrongful approvals | Wrongful declines |
|---|---|---|---|---|
| Payout tool | $383 | $107 | $1,090 | $500 |
| Qdrant | $67 | $47 | $0 | $2,500 |
| Word lookup | $50 | $28 | $0 | $4,500 |
| Qdrant + notes | $33 | $0 | $0 | $2,000 |
| Notes kept | $250 | $0 | $0 | $750 |
| Sub-agent | $50 | $28 | $0 | $500 |

Without the memory, the agent approves claims that a storm or strike announced before the purchase excludes, and refers claims it can't settle. With the memory, those mistakes go. What's left is mostly wrongful declines, at $1,500 each.

### Decisions are final: the second thought

There were 20 wrongful declines in the memory experiments, and 19 of them were on three claims: CLM-1302, CLM-1304 and CLM-1306. On each, a storm was named or a strike announced *after* the policy was bought. The fact was found every time.

The rationale shows the slip as the agent writes it. Here is CLM-1304, from Qdrant's run 1:

> The cabin crew strike was publicly announced on August 5, 2026, before this policy was purchased on July 28? Actually, the announcement date is after purchase; therefore this exclusion does not apply.

The decision it recorded with that rationale was decline. In 15 of the 20 wrongful declines, the agent then tried to change the decision, 14 times to exactly the handler's answer. `record_decision` refused every time, because decisions are final. All 15 refused changes in the 425 answers were attempts to undo a wrongful decline. Oleg Mosiichuk found this pattern in his run.

Scoring the same answers as if the last attempt had been accepted:

| | Tried to change, a run | Accuracy | If accepted | Cost of mistakes a run | If accepted |
|---|---|---|---|---|---|
| Qdrant | 1.33 | 86.7% | 92.0% | $2,614 | $614 |
| Word lookup | 2.33 | 82.7% | 92.0% | $4,578 | $1,078 |
| Qdrant + notes | 1.00 | 92.0% | 94.7% | $2,033 | $633 |
| Sub-agent | 0.33 | 93.3% | 94.7% | $578 | $78 |

This is an estimate: an agent that knows it can revise might also behave differently. Oleg tested the real thing. He made `record_decision` keep a draft until the agent finishes, and in one run his cost of mistakes fell from $3,000 to $150. His accuracy fell from 0.92 to 0.88, because three claims were referred needlessly. So the change works, and it does change how the agent behaves.

A rationale that corrects itself with "Actually", "Wait" or "Correction" is a warning sign. Such answers were wrong in 17 of 36 cases across the experiments, and in 8 of the word lookup's 11.

### Challenge 1: searching by words against searching by meaning

Every query the agents wrote, sent to both searches again. "Found" is the share of claims where any of the run's queries brought back the needed fact in the top 10; "first" is where one ranked it first.

| Whose queries | Found, by meaning | Found, by words | First, by meaning | First, by words | MRR, by meaning | MRR, by words |
|---|---|---|---|---|---|---|
| Qdrant agent | 100% | 100% | 96.5% | 91.2% | 0.982 | 0.953 |
| Word lookup agent | 100% | 100% | 96.5% | 91.2% | 0.982 | 0.956 |
| Qdrant + notes | 100% | 100% | 96.5% | 94.7% | 0.982 | 0.974 |
| Sub-agent | 100% | 100% | 96.5% | 93.0% | 0.982 | 0.958 |
| **The claim's summary, with no agent** | **89.5%** | **73.7%** | 63.2% | 63.2% | 0.695 | 0.675 |

The agents' queries name the thing they need, such as "Halden Arc product device electronics classification" or "Sentinel Wall Cayo Brava Honduras Brava Divers dive depth". That's all a word lookup needs.

The summary is how a pipeline without an agent might search. Arthur JZL tried it. The summary often describes a device without naming it, such as "Checked bag lost on the way to Tokyo" or "Backpack stolen on the Paris metro". Then the word lookup finds nothing: it missed five of the seven device claims, CLM-1307 to CLM-1313. Qdrant missed two and ranked four more between 3rd and 10th.

So the word lookup agent's extra wrongful declines don't come from its search. Eight of its nine were on the three dated claims, with the fact found. With the last attempt accepted, both searches score 92.0%. I can't say why the word lookup agent slipped more often, and the interval for the cost of mistakes, -$7 to +$4,495, just includes zero.

### Challenge 2: the customer notes

**The agent used them.** It read the notes on 100% of claims and added a note on 100%. Its notes say what it decided in 97% to 100% of claims.

Two notes are worth reading. In two runs, CLM-1304's note argues for the approval that the agent couldn't record, under a recorded decline. A note that contradicts the recorded decision is a sign that the decision needs a second look.

The notes have two flaws that classmates found too:
- **The agent sometimes adds its own date** at the start of a note, beside the tool's, as Arthur JZL found: "2026-11-21, claims agent: 2026-07-30: Referred CLM-1321 ...". Its dates are made up.
- **A wrong decision becomes a wrong note** for the next handler, as Hugo Volckaert and James Horrigan said. James's agent wrote the date slip itself into a note.

**The misleading notes changed nothing.** Every experiment that read the notes, in every run, got these claims right:

- CLM-1305 (the loyal customer): declined
- CLM-1324 (the police report that "arrived"): referred
- CLM-1319 (the rude customer): approved

The one exception was a sub-agent run that referred CLM-1319 over whether a jet-boat ride is an adventure sport. Qdrant, with no notes, made the same referral in the same round, and so did Darko Martinovic's agent. My misleading notes put pressure on the agent. James Horrigan's gave it a wrong fact instead: a note of something a handler had told the customer wrongly, on CLM-1309. His agent ignored it and followed the wording.

**Kept notes pass the answers on.** With the notes kept, run 2 agreed with the run before it on 76% of claims, against 96% with a reset. That isn't the agent copying its earlier answers: Notes kept was wrong 11 times, and 10 of those were referrals like this one, on CLM-1307:

> The customer notes already record CLM-1307 as approved for $562.95, matching this same lost-baggage incident, but the claim is still marked open. Refer for review of the prior handling/payment to avoid making a duplicate payment.

There were 4 such referrals in run 2 and 6 in run 3, growing as the notes piled up. Andras Nemes and Darko Martinovic each ran one run with kept notes, and saw the same referrals. Andras's agent also approved CLM-1302 "because prior customer notes record this same claim as approved", which was right, but copied. The second run in my chain shows the effect growing. The other mistake was a decline of CLM-1302, whose note says the file has "an inconsistent apparent earlier approval note".

The leak also worked the other way once. On CLM-1304, in run 1 of Qdrant + customer notes, the agent's correction was refused, so it wrote the correction into the note:

> the recorded rationale incorrectly says the strike announcement preceded policy purchase

Notes kept read that note in runs 2 and 3, and approved. Qdrant + customer notes, starting from fresh notes, declined CLM-1304 in all three runs. Hugo Volckaert's agent wrote the same kind of correction on CUS-3304.

Notes kept has a lower cost of mistakes, $1,000 a run, than the same runs with a reset, at $2,300. That's only because its mistakes are $50 referrals instead of $1,500 declines.

### Stretch: the research sub-agent

| | Sub-agent |
|---|---|
| The needed fact found / passed on to the UnderCover agent | 100% / 100% |
| Research calls a claim | 2.4 |
| Share of model calls made by the sub-agent | 53% |
| Input read from the provider's cache | 78%, against 90% for Qdrant + notes |

Every sub-agent call starts a fresh conversation, which may be why less of its input came from the cache. As a result it cost 67% more, although it made 2.2 times as many model calls.

It made one wrongful decline in 75 claims, against four for Qdrant + notes. That's not a clear difference, but one pattern is worth testing.

Hugo Volckaert's sub-agent fixed his wrongful declines on CLM-1304 and CLM-1306 in his one run. He suggested that the research becomes a step of its own, so the agent compares the dates before it records anything. Andras Nemes saw his agent learn to pass on the policy date, "so the sub-agent could check the timing itself". My runs support them. On 8 of the 18 storm and strike claims, the UnderCover agent's request gave the date the policy was bought. Each time, the sub-agent answered "before" or "after" in so many words, and all 8 were decided right. Here is CLM-1304, run 2:

> Request: Look up whether the Meridian cabin crew strike causing the August 12, 2026 flight cancellation was publicly known before policy purchase on July 28, 2026.
>
> Reply: The memory says the strike was announced **after** the policy purchase date ...

The 10 claims where the request didn't give the date got 9 right. The one wrongful decline was among them.

### How this compares with other students' runs

Accuracy in the other Week 2 Day 2 contributions, all on GPT-6 Luna at `low` except Darko's:

| | Runs | No memory | Qdrant | Word lookup | Notes | Notes kept | Sub-agent |
|---|---|---|---|---|---|---|---|
| Andras Nemes | 3 | - | 93% | 93% | 99% | 88% (1 run) | 85% |
| Arthur JZL | 3 | 57% | 96% | 92% | 97% | - | 96% |
| Darko Martinovic, on Haiku 5.5 | 2 | 38% | 96% | 100% | 96% | 84% (1 run) | 96% |
| Hugo Volckaert | 1 | 52% | 88% | 92% | 84%, with the word lookup | - | 96% |
| James Horrigan | 1 | 60% | 100% | 92% | 96%, with the word lookup | - | - |
| Oleg Mosiichuk | 1 | 64% | 92% | - | - | - | - |
| **Mine** | 3 | 54.7% | 86.7% | 82.7% | 92.0% | 78.0% (2 runs) | 93.3% |

VerBodh ran the 5 tiny claims once, and Steve Higgins's notebook was saved without its results.

**Where we agree:**
- **Retrieval isn't the problem.** Everyone's searches found the needed fact on every claim, with either search.
- **The word lookup is as good as Qdrant, within the noise.** It came out ahead in two notebooks, level in one and behind in three, mine included.
- **The word lookup is faster.** It took 7.7 seconds a claim against 14.7 for Andras, 9.4 against 16.7 for Darko, and 7.3 against 10.7 for me.
- **The summary-as-query test agrees.** Arthur's retrieval test with the claim's summary as the query gave 74% by words and 89% by meaning. Mine gave 73.7% and 89.5%, with a different word lookup.
- **No one showed a clear gain from the notes, and no misleading note misled the agent.** James Horrigan's two improved claims had starting notes that pointed toward the handler's decision, as he said.
- **Kept notes made the agent refer claims to avoid paying twice,** in all three notebooks that tried it.
- **The sub-agent costs more in every notebook.** Against an agent that searches for itself, that's 17% more for Darko, about twice as much for Andras, Arthur and Hugo, and 67% more for me.
- **The same judgment calls come up:**
  - the shared taxi receipt on CLM-1323 (Darko)
  - the jet-boat ride on CLM-1319 (Darko)
  - the wallet without a receipt on CLM-1308 (Arthur)

**Where my runs differ:**
- **My Qdrant agent is at the low end, at 86.7%.** The difference is the slip over dates. Arthur's Qdrant agent made no wrongful declines in three runs, a cost of mistakes of $67 a run. Mine made five, at $2,614 a run. Hugo and Oleg each declined CLM-1304 and CLM-1306 in their single runs. Same model, same setting and the lab's prompts, so I can't explain the gap. It's a reminder that two students' runs of the same experiment can differ by more than the experiments do.
- **The sub-agent's accuracy varies most:** from 85% for Andras, whose agent made needless referrals, to 96% for Arthur, Darko and Hugo. Andras suggested that facts reported back by an assistant seem to carry less weight than facts the agent found itself. My runs don't show that: my sub-agent made 3 needless referrals in 75 claims.

### Conclusions

1. **Give the agent the memory.** It's worth 32 points of accuracy for less than half a cent a run. But don't expect the cost of mistakes to fall until the slip over dates is dealt with.
2. **The biggest gain on offer is the second thought.** The agent catches most of its own wrongful declines, and the rule that decisions are final keeps them. These are the next things I'd try:
   - letting a decision be a draft until the agent finishes, as Oleg Mosiichuk did
   - a check that flags a decision its own rationale argues against
   - asking the agent to compare the dates before it records the decision

   Day 1 showed the same pattern, a rationale arguing against the decision recorded with it, but only twice.
3. **For this knowledge base, a word lookup is enough when an agent writes the queries,** and it's faster. Searching by meaning is safer when the query might describe the thing instead of naming it.
4. **Reset the notes before every run of an evaluation; keep them in production.** In an evaluation, kept notes show each run the answers to the same claims. In production, a note that says a claim was already paid should make the agent stop, and here it did. Whether the notes help needs a set with several claims for each customer, as several classmates said.

   Two classmates' designs would make the notes safer:
   - Steve Higgins told his agent to leave the decision and the claim id out of its notes. That would stop notes from passing answers between runs, though it would also stop the CLM-1304 correction.
   - James Horrigan suggested that code, not the model, writes the decision into the notes. Then a wrong summary couldn't mislead the next handler.
5. **Keep the sub-agent out for now.** It cost 67% more and was slower, for a gain these runs can't show. If I tried it again, I'd have the UnderCover agent always send the purchase date with the request.

### Limits

- 25 claims and three runs, two for Notes kept, on one model at one reasoning setting.
- Each customer has one claim, so the notes were tested only with my seven starting notes and the agent's own notes. Three misleading notes is a small test.
- The second scoring is an estimate. An agent that can revise its decision might reason differently from the start.
- The before-and-after pattern in the sub-agent's replies comes from 18 claims, and the checks for it and for notes that mention the decision are text matches.
- Qdrant's time includes starting its server for each search, as in the lab. That's most of the word lookup's speed advantage.
- The model costs are what OpenRouter reported.

### Credits

Ideas from other students' Week 2 Day 2 contributions:

- **Oleg Mosiichuk:**
  - the memory raising accuracy but not lowering the cost of mistakes
  - decisions being final
  - a decision kept as a draft until the agent finishes, and his run of it
- **Andras Nemes:**
  - the answer to the `N_TIMES` question I started from, in nearly his words
  - a run with kept notes
  - the three things that need care in a sub-agent: its searches and cost hidden from the scores, its seeing only the request, and its trace nested in Langfuse
  - the agent passing on the policy date
- **Darko Martinovic:**
  - a run with kept notes, with the same referrals to avoid paying twice
  - the word lookup with the best accuracy, in his runs on Haiku
- **Hugo Volckaert:**
  - CLM-1304 and CLM-1306 declined with the fact found
  - the sub-agent making the date comparison a step of its own
  - the correction in CUS-3304's note
  - a wrong decision becoming a wrong note
- **Arthur JZL:**
  - the claim's summary as the query
  - the agent's own dates in its notes
- **James Horrigan:**
  - a misleading note that gives a wrong fact
  - a note with the date slip in it
  - writing the decision into the notes with code
- **Steve Higgins:** notes that leave out the decision and the claim id
- **VerBodh:** a note filed under a claim id